<br>

# RAJs, Circunscrição Judiciária e Comarcas

Por meio do site [Quem Somos](https://www.tjsp.jus.br/QuemSomos/QuemSomos/RegioesAdministrativasJudiciarias) raspei as informações

<br>

Michel Metran\
Data: X.X.2020\
Atualizado em: 09.06.2025


In [ ]:
# !pip3 install traquitanas
# !pip3 install lxml

In [ ]:
import re
import urllib.request

import pandas as pd
from lxml import html
from functions import keep_numbers
from paths import output_path_tab

In [ ]:
# from bs4 import BeautifulSoup
# from selenium.webdriver.common.by import By

# from sp_tjsp_divadmin.my_driver import Driver
# from sp_tjsp_divadmin.my_functions import find_text_between_parenthesis
# from sp_tjsp_divadmin.my_paths import adds_path, driver_path, logs_path

<br>

---

## RAJs

Pego os dados das RAJs


In [ ]:
# Go TJSP
URL = 'https://www.tjsp.jus.br/QuemSomos/QuemSomos/RegioesAdministrativasJudiciarias'

In [ ]:
content = urllib.request.urlopen(url=URL).read()
tree = html.fromstring(content)
tree

In [ ]:
list_divs = tree.xpath("//div[contains(@style, 'background')]")
list_divs

In [ ]:
list_dfs_rajs = []
list_dfs_cjs = []

for div in list_divs:
    # A partir da div, pega os "p"
    list_p = div.xpath('.//p')
    raj = list_p[0].text_content().strip()
    raj_list = re.split(pattern=r'[-–]', string=raj, maxsplit=0)
    raj_num = raj_list[0].strip()
    raj_regiao = raj_list[1].strip()
    juiz = list_p[1].text_content().strip().split(':')[1].strip()
    email = re.sub('[()]', '', list_p[2].text_content().strip())

    # Para cada div, pega o primeiro ul que encontramos
    list_ul = div.xpath('./..//ul')[0]
    list_ul = div.getnext()
    list_li = list_ul.xpath('.//li')

    dict_raj = {
        'raj_nome': raj,
        'raj_sigla': raj_num,
        'raj_regiao': raj_regiao,
        'juiz_diretor_nome': juiz,
        'juiz_diretor_email': email,
    }
    list_dfs_rajs.append(dict_raj)

    # Circunscrição Judiciária
    list_cjs = [x.text_content() for x in list_li]
    df = pd.DataFrame(data=list_cjs, columns=['comarca_cirscunscricao'])
    df['raj_sigla'] = raj_num
    list_dfs_cjs.append(df)

# ddd
df = pd.concat(list_dfs_cjs, ignore_index=True)
df.info()
df.head()

In [ ]:
df_raj = pd.DataFrame(list_dfs_rajs)
df_raj['raj_nome'] = df_raj['raj_nome'].str.replace('–', '-')
df_raj['id_raj'] = df_raj['raj_sigla'].apply(lambda x: keep_numbers(x))
df_raj['id_raj'] = df_raj['id_raj'].astype(int)
df_raj = df_raj.sort_values(by='id_raj', ascending=True)
df_raj = df_raj.reset_index(drop=True)

df_raj = df_raj[
    [
        # RAJ
        'id_raj',
        'raj_nome',
        'raj_sigla',
        'raj_regiao',
        'juiz_diretor_nome',
        'juiz_diretor_email',
    ]
]

df_raj

In [ ]:
filename = "RAJs"

df_raj.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)

df_raj.to_excel(
    excel_writer=output_path_tab / f"{filename}.xlsx",
    index=False,
    sheet_name=f"{filename}",
)

<br>

---

## Comarcas e CJs


In [ ]:
df.head()

In [ ]:
# ddd
df_cj = df.copy()
df_cj[['comarca', 'cj_sigla']] = df_cj['comarca_cirscunscricao'].str.rsplit(
    '-', n=1, expand=True
)

df_cj['cj_sigla'] = df_cj['cj_sigla'].str.strip()


# Id RAJ
df_cj['id_raj'] = df_cj['raj_sigla'].apply(lambda x: keep_numbers(x))
df_cj['id_raj'] = df_cj['id_raj'].astype(int)


# Id CJ
df_cj['id_cj'] = df_cj['cj_sigla'].apply(lambda x: keep_numbers(x))
df_cj.loc[df_cj['id_cj'] == '', 'id_cj'] = '0'
df_cj['id_cj'] = df_cj['id_cj'].astype(int)


df_cj['cj_nome'] = df_cj['cj_sigla'].replace(
    'CJ', 'Circunscrição Judiciária', regex=True
)
df_cj['cj_nome'] = df_cj['cj_nome'].str.strip()

df_cj = df_cj[
    [
        # CJ
        'id_cj',
        'cj_sigla',
        'cj_nome',
        #'comarca_cirscunscricao',
        # Comarca
        #'comarca',
        'id_raj',
    ]
]

df_cj = df_cj.drop_duplicates()
df_cj = df_cj.sort_values(by='id_cj')
df_cj = df_cj.reset_index(drop=True)
df_cj = df_cj.drop_duplicates()

# Results
df_cj.to_clipboard(index=False)
df_cj.info()
df_cj.head()

In [ ]:
filename = 'Circunscrições Judiciárias'

df_cj.to_csv(
    path_or_buf=output_path_tab / f'{filename}.csv',
    index=False,
)

df_cj.to_excel(
    excel_writer=output_path_tab / f'{filename}.xlsx',
    sheet_name=f'{filename}',
    index=False,
)

<br>

---

## CJs


In [ ]:
df_cjs = df_cj[
    [
        # CJ
        'id_cj',
        'cj_nome',
        'cj_sigla',
        'id_raj',
    ]
]

# df_cjs.loc[df_cjs['cj_id'] == '', 'cj_id'] = '0'
# df_cjs['cj_id'] = df_cjs['cj_id'].copy()
# df_cjs['cj_id'] = df_cjs['cj_id'].astype('int')
df_cjs = df_cjs.drop_duplicates()
df_cjs = df_cjs.sort_values(by='id_cj')
df_cjs = df_cjs.reset_index(drop=True)

# Results

df_cjs.info()
df_cjs.head()

In [ ]:
filename = "Circunscrições Judiciárias"

df_cjs.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)

df_cjs.to_excel(
    excel_writer=output_path_tab / f"{filename}.xlsx",
    sheet_name=f"{filename}",
    index=False,
)

<br>

---

## Comarcas


In [ ]:
df.head()

In [ ]:
# ddd
df_comarca = df.copy()
df_comarca[['comarca', 'cj_sigla']] = df_comarca[
    'comarca_cirscunscricao'
].str.rsplit('-', n=1, expand=True)

df_comarca['cj_sigla'] = df_comarca['cj_sigla'].str.strip()


# Id RAJ
# df_comarca['id_raj'] = df_comarca['raj_sigla'].apply(lambda x: keep_numbers(x))
# df_comarca['id_raj'] = df_comarca['id_raj'].astype(int)


# Id CJ
df_comarca['id_cj'] = df_comarca['cj_sigla'].apply(lambda x: keep_numbers(x))
df_comarca.loc[df_comarca['id_cj'] == '', 'id_cj'] = '0'
df_comarca['id_cj'] = df_comarca['id_cj'].astype(int)


df_comarca['cj_nome'] = df_comarca['cj_sigla'].replace(
    'CJ', 'Circunscrição Judiciária', regex=True
)
df_comarca['cj_nome'] = df_comarca['cj_nome'].str.strip()

df_comarca = df_comarca[
    [
        # CJ
        'comarca',
        'id_cj',
        #'cj_sigla',
        #'cj_nome',
        #'comarca_cirscunscricao',
        # Comarca
        #'id_raj',
    ]
]

df_comarca = df_comarca.drop_duplicates()


# Renomear
df_comarca = df_comarca.rename(
    {
        'comarca': 'comarca_tjsp',
    },
    axis='columns',
)

# Ordena
df_comarca = df_comarca.iloc[
    df_comarca['comarca_tjsp'].str.normalize('NFKD').argsort()
]
df_comarca = df_comarca.reset_index(drop=True)

# Results
df_comarca.to_clipboard(index=False)
df_comarca.info()
df_comarca.head(20)

In [ ]:
filename = "Comarcas e CJs"

df_comarca.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)

df_comarca.to_excel(
    excel_writer=output_path_tab / f"{filename}.xlsx",
    index=False,
    sheet_name=f"{filename}",
)

In [ ]:
comarcas = df_comarca['comarca_tjsp']

print(f'São {len(set(comarcas))} comarcas')